## ধাপ ০: Anchor বানানো

**কখন:** training শুরুর আগে, **একবারই**। পুরো training-এ একই anchor ব্যবহার হয়।

**কী লাগে:** শুধু তিনটা জিনিস, কোনো ছবি, network বা ground truth লাগে না:
- ছবির সাইজ (যেমন 640×640)
- feature map-গুলোর stride (যেমন 8, 16, 32)
- প্রতিটা stride-এ কোন সাইজের anchor (যেমন 16 আর 32 px, 64 আর 128 px ...)

**কী হয়:** প্রতিটা feature map-এর প্রতিটা ঘরের center-এ নির্দিষ্ট সাইজের box বসানো হয়।

**Output:** `anchors [P, 4]`, center form `(cx, cy, w, h)`, 0-1-এ normalized। 640 ছবিতে P = 16800।

**RetinaFace-এ:** `PriorBox(cfg, image_size).forward()`

**Inference-এ:** ছবির সাইজ বদলালে anchor আবার বানাতে হয়, কারণ grid-এর সাইজ ছবির ওপর নির্ভর করে।

## ধাপ ১: Data লোড

**কখন:** প্রতিটা batch-এ।

**কী হয়:** ছবি পড়া হয়, augmentation হয় (random crop, flip, resize), তারপর GT box-গুলো ছবির সাইজ দিয়ে ভাগ করে 0-1-এ আনা হয়।

**Output:**
- `images [B, 3, 640, 640]`
- `targets`: প্রতিটা ছবির নিজের তালিকা `[num_objects, 4 + 1]` (box + label)। একেক ছবিতে object-এর সংখ্যা আলাদা, তাই এটা একটা tensor না, তালিকা।

## ধাপ ২: Forward pass (শুধু network)

**কখন:** প্রতিটা batch-এ।

**কী হয়:**
1. **Backbone** ছবি থেকে feature বের করে, কয়েকটা ভিন্ন সাইজের feature map দেয় (stride 8, 16, 32 মানে 80×80, 40×40, 20×20)।
2. প্রতিটা feature map-এর ওপর একটা ছোট **head** (conv layer) চলে। প্রতিটা ঘরে K টা anchor থাকলে head দেয়:
   - class head: প্রতি ঘরে `K × C` সংখ্যা (C = class সংখ্যা, background সহ)
   - box head: প্রতি ঘরে `K × 4` সংখ্যা
3. সব feature map-এর output reshape করে পরপর জোড়া লাগানো হয়।

**Output:**
- `class_pred [B, P, C]`
- `box_pred [B, P, 4]`, প্রতিটা `[dx, dy, dw, dh]`

**মূল কথা:** জোড়া লাগানোর ক্রম আর ধাপ ০-এর anchor বানানোর ক্রম হুবহু এক। তাই `box_pred`-এর n নম্বর সারি = n নম্বর anchor-এর prediction। এখানে network কোনো anchor বা GT দেখে না, শুধু সংখ্যা দেয়।

## ধাপ ৩: Target বানানো (network ছাড়া)

**কখন:** প্রতিটা batch-এ, batch-এর **প্রতিটা ছবির জন্য আলাদা করে**।

**কেন প্রতিবার?** Anchor তো একই, কিন্তু augmentation-এর কারণে একই ছবির GT box প্রতি epoch-এ আলাদা জায়গায় থাকে (random crop)। তাই কোন anchor positive, সেটা প্রতিবার নতুন করে হিসাব করতে হয়।

**কী হয় (প্রতিটা ছবির জন্য):**
1. **IoU টেবিল:** anchor-কে corner form-এ এনে সব GT বনাম সব anchor-এর IoU, shape `[num_gt, P]`।
2. **প্রতিটা anchor-এর সেরা GT:** কলাম বরাবর max, shape `[P]`।
3. **প্রতিটা GT-এর সেরা anchor:** সারি বরাবর max। এই anchor-গুলোকে জোর করে positive বানানো হয়, যাতে কোনো object বাদ না যায়।
4. **Label:** IoU threshold-এর নিচে হলে background (0), নাহলে ওই GT-এর class।
5. **Encode:** প্রতিটা anchor-এর জন্য তার GT-এর দিকে সঠিক offset `[dx, dy, dw, dh]`।

**Output:**
- `class_target [B, P]` (0 = background, 1, 2 ... = class)
- `box_target [B, P, 4]`

**RetinaFace-এ:** `match()`, যেটা `MultiBoxLoss`-এর ভেতর থেকে প্রতিটা ছবির জন্য call হয়।

## ধাপ ৪: Loss

**কখন:** প্রতিটা batch-এ।

**কী হয়:**
1. **Positive mask:** `pos = class_target > 0`
2. **Box loss:** শুধু positive anchor-এর জন্য, `SmoothL1(box_pred[pos], box_target[pos])`। Background-এর box-এর কোনো মানে নেই, তাই বাদ।
3. **Class loss:**
   - প্রথমে প্রতিটা anchor-এর cross entropy হিসাব
   - Negative অনেক বেশি (প্রায় সব anchor), তাই **hard negative mining**: negative-দের মধ্যে যাদের loss সবচেয়ে বেশি (network যাদের ভুল করে object ভাবছে), শুধু তাদের নেওয়া, positive-এর কয়েক গুণ (RetinaFace-এ ৭ গুণ)
   - তারপর `CrossEntropy(positive + বাছাই করা negative)`
4. **Normalize:** দুটো loss-কেই positive anchor-এর সংখ্যা দিয়ে ভাগ।
5. **Total:** `loss = class_loss + α × box_loss` (RetinaFace-এ α = 2)

## ধাপ ৫: Backward আর update

**কখন:** প্রতিটা batch-এ।

`loss.backward()` gradient হিসাব করে, `optimizer.step()` network-এর weight বদলায়। তারপর পরের batch-এ আবার ধাপ ১ থেকে, কিন্তু ধাপ ০ আর হয় না।

## সময়ের হিসাবে এক নজরে

| কাজ | কখন | কতবার |
|---|---|---|
| Anchor বানানো | training শুরুর আগে | **একবার** |
| Data লোড + augmentation | প্রতি batch | প্রতি batch-এ একবার |
| Forward pass | প্রতি batch | প্রতি batch-এ একবার |
| Target বানানো (`match`) | প্রতি batch | batch-এর **প্রতিটা ছবির জন্য** একবার |
| Loss | প্রতি batch | প্রতি batch-এ একবার |
| Backward + update | প্রতি batch | প্রতি batch-এ একবার |

ধাপ ২ (network) আর ধাপ ৩ (target) একে অপরের ওপর নির্ভর করে না, দুটোই আলাদা আলাদা হিসাব হয়। ধাপ ৪-এ এসে প্রথমবার দুটো মুখোমুখি হয়: network কী বলল বনাম কী বলা উচিত ছিল।